In [ ]:
from pyqcu import cuda, tools
from pyqcu.cuda import qcu, params, argv, set_ptrs
params[cuda._LAT_X_] = 8
params[cuda._LAT_Y_] = 8
params[cuda._LAT_Z_] = 8
params[cuda._LAT_T_] = 8
params[cuda._LAT_XYZT_] = params[cuda._LAT_X_] * \
    params[cuda._LAT_Y_]*params[cuda._LAT_Z_]*params[cuda._LAT_T_]
params[cuda._GRID_X_], params[cuda._GRID_Y_], params[cuda._GRID_Z_], params[
    cuda._GRID_T_] = tools.give_grid_size()
params[cuda._PARITY_] = 0
params[cuda._NODE_RANK_] = cuda.rank
params[cuda._NODE_SIZE_] = cuda.size
params[cuda._DAGGER_] = 0
params[cuda._MAX_ITER_] = 1000
params[cuda._DATA_TYPE_] = cuda._LAT_C128_
params[cuda._SET_INDEX_] = 0
params[cuda._SET_PLAN_] = 0
params[cuda._MG_X_] = 1
params[cuda._MG_Y_] = 1
params[cuda._MG_Z_] = 1
params[cuda._MG_T_] = 1
params[cuda._LAT_E_] = 24
params[cuda._VERBOSE_] = 1
params[cuda._SEED_] = 42
argv[cuda._MASS_] = 0.05
argv[cuda._TOL_] = 1e-12
argv[cuda._SIGMA_] = 0.1
#############################
print(params)
print(argv)
print(set_ptrs)

In [ ]:

gauge = io.give_none_gauge(params, save=False)
fermion_in = io.give_none_fermion_in(params, save=False)
wilson_fermion_out = io.give_none_fermion_out(params, save=False)
clover_fermion_out = io.give_none_fermion_out(params, save=False)
#############################
gauge = cp.zeros_like(gauge)
fermion_in = cp.ones_like(fermion_in)
wilson_fermion_out = cp.zeros_like(fermion_in)
clover_fermion_out = cp.zeros_like(fermion_in)
#############################
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyGaussGaugeQcu(gauge, set_ptrs, params)
qcu.applyEndQcu(set_ptrs, params)
#############################
params[cuda._VERBOSE_] = 1
params[cuda._SET_INDEX_] += 1
params[cuda._SET_PLAN_] = 1
#############################
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyWilsonBistabCgQcu(
    wilson_fermion_out, fermion_in, gauge, set_ptrs, params)
qcu.applyEndQcu(set_ptrs, params)
#############################
clover_ee = io.give_none_clover(params, save=False)
clover_ee_inv = cp.zeros_like(clover_ee)
clover_oo = cp.zeros_like(clover_ee)
clover_oo_inv = cp.zeros_like(clover_ee)
#############################
params[cuda._VERBOSE_] = 1
params[cuda._SET_INDEX_] += 1
params[cuda._SET_PLAN_] = 2
params[cuda._PARITY_] = 0
#############################
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyCloversQcu(clover_ee, clover_ee_inv, gauge, set_ptrs, params)
qcu.applyEndQcu(set_ptrs, params)
#############################
params[cuda._VERBOSE_] = 1
params[cuda._SET_INDEX_] += 1
params[cuda._SET_PLAN_] = 2
params[cuda._PARITY_] = 1
#############################
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyCloversQcu(clover_oo, clover_oo_inv, gauge, set_ptrs, params)
qcu.applyEndQcu(set_ptrs, params)
#############################
params[cuda._VERBOSE_] = 1
params[cuda._SET_INDEX_] += 1
params[cuda._SET_PLAN_] = 1
#############################
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyCloverBistabCgQcu(clover_fermion_out, fermion_in,
                           gauge, clover_ee, clover_oo, clover_ee_inv, clover_oo_inv,  set_ptrs, params)
qcu.applyEndQcu(set_ptrs, params)
#############################
if os.path.exists("_gauge-test8.h5") and os.path.exists("_fermion_in-test8.h5") and os.path.exists("_wilson_fermion_out-test8.h5") and os.path.exists("_clover_fermion_out-test8.h5"):
    io.grid_xxxtzyx2hdf5_xxxtzyx(
        wilson_fermion_out, params, "_wilson_fermion_out-test8.h5")
    io.grid_xxxtzyx2hdf5_xxxtzyx(
        clover_fermion_out, params, "_clover_fermion_out-test8.h5")
    print(
        f"@cp.linalg.norm(wilson_fermion_out-_wilson_fermion_out)/cp.linalg.norm(wilson_fermion_out):{cp.linalg.norm(wilson_fermion_out-_wilson_fermion_out)/cp.linalg.norm(wilson_fermion_out)}@")
    print(
        f"@cp.linalg.norm(clover_fermion_out-_clover_fermion_out)/cp.linalg.norm(clover_fermion_out):{cp.linalg.norm(clover_fermion_out-_clover_fermion_out)/cp.linalg.norm(clover_fermion_out)}@")
#############################